# K-Merge Real Benchmark: merging vs sharing on real data

**What this notebook tests.** K-Merge keeps K adapter slots on the device. When a new task's LoRA adapter arrives and the slots
are full, K-Merge averages it into the most similar slot. This notebook rebuilds K-Merge's own kind of benchmark with **real
data** and compares, at **exactly the same storage**, how a slot should hold several tasks:

| Family | Methods | Idea |
|---|---|---|
| **Merging** (one adapter per slot) | `kmerge` (K-Merge Eq. 8), `linear`, `ties`, `dare_ties` (K-Merge's baselines), `regmean` (strongest known merge, needs 32 unlabeled inputs) | average / combine the tasks into one adapter |
| **Sharing** (task-conditional slot; K-Merge already knows the task of every input) | `split` (each task keeps its top r/M directions: the TSV-C idea), `kshare` (shared basis + small per-task core), **`adaptive`** and **`adaptive_fair`** (NEW: each task gets as many directions as it needs, decided per layer), **`wsplit`** and **`wadaptive`** (NEW, activation-aware: each task keeps the directions that change its outputs most on 32 of its own unlabeled inputs, the same data RegMean uses) | keep a small private part of every task inside the slot |

**Tasks (12 = 3 problem types x 4 languages, as in K-Merge):**

| Problem type | English | German | Spanish | Italian |
|---|---|---|---|---|
| Grammar correction (real learner errors) | Write & Improve (BEA-2019) | Falko-MERLIN | COWS-L2H | MERLIN |
| Summarization | SAMSum | SAMSum, translated with OPUS-MT | same | same |
| Question answering | SQuAD | SQuAD, translated with OPUS-MT | same | same |

Model: Qwen2.5-1.5B-Instruct (as in K-Merge), LoRA rank 32 on q, k, v, o (K-Merge's setting). Tasks arrive one at a time
(random order and K-Merge's worst case: same problem type in a row), K = 2 and 4 slots, every method uses the same slot
assignment (K-Merge's cosine rule), so the only difference is how a slot stores its tasks. Score = each task's metric
divided by its own single-task adapter (K-Merge's normalised score); significance by paired bootstrap over test examples.

### How to run on Kaggle
1. *Settings* → **Accelerator: GPU T4 x2**, **Internet: ON** (downloads the model, the datasets and the translation models).
2. `RUN_MODE = "SMOKE"` first (about 15 min), then `"QUICK"` (seed 0, ~4.5 h), then `"FULL_A"` (seeds 0-1) and `"FULL"`
   (seeds 0-2), each resuming from the previous version (one run per step keeps every run under Kaggle's 12-hour limit).
3. **Resume / reuse:** *Add Input* → *Your Work* → this notebook; finished work (data, translations, adapters, evaluations)
   from that earlier version is copied in and skipped.
4. Send back the text under `SEND THIS BACK` (also `RESULTS_SUMMARY.txt`) and `results.zip`.

Dataset licences: Write & Improve (non-commercial research), Falko-MERLIN and MERLIN (CC BY-SA 4.0), COWS-L2H (CC BY 4.0),
SAMSum (CC BY-NC-ND 4.0; translations are used internally, not redistributed), SQuAD (CC BY-SA 4.0).

In [ ]:
# ============================================================================
# 1. SETUP AND CONFIG
# ============================================================================
import os, sys, re, json, math, time, random, itertools, difflib, subprocess, zipfile, gc, hashlib, heapq, tarfile, io
import contextlib
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less GPU memory fragmentation (T4: 15 GB)
import urllib.request
from collections import OrderedDict, Counter, defaultdict


def _internet_ok(url="https://huggingface.co", timeout=8):
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except Exception:
        return False


INTERNET = _internet_ok()


def _ensure(pkg, pip_name=None):
    try:
        __import__(pkg)
        return True
    except ImportError:
        if not INTERNET:
            return False
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg], capture_output=True, text=True)
        try:
            __import__(pkg)
            return True
        except ImportError:
            print(r.stdout[-1500:], r.stderr[-1500:])
            return False


if not _ensure("peft"):
    raise RuntimeError("PRE-FLIGHT FAILED: the 'peft' library is missing and could not be installed.\n"
                       "FIX: right panel -> Settings -> Internet -> On, then Run All again.")
for _p in ("sentencepiece", "pyarrow"):
    _ensure(_p)


def _fix_torchao():
    """Kaggle ships torchao 0.10, but recent peft refuses to build LoRA layers while an older torchao is installed.
    This notebook never uses torchao, so remove it (works without a restart, torchao itself is never imported)."""
    try:
        from peft.import_utils import is_torchao_available
    except Exception:
        return
    try:
        is_torchao_available()
        return
    except ImportError as e:
        print(f"PRE-FLIGHT: {e}\n            -> uninstalling torchao (this notebook does not need it)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], capture_output=True, text=True)
    import importlib
    importlib.invalidate_caches()
    if hasattr(is_torchao_available, "cache_clear"):
        is_torchao_available.cache_clear()
    try:
        is_torchao_available()
        print("PRE-FLIGHT: torchao problem fixed")
    except ImportError:
        raise RuntimeError("PRE-FLIGHT FAILED: an incompatible torchao is installed and could not be removed.\n"
                           "FIX: add a first cell containing   !pip uninstall -y torchao   then Restart & Run All.")


_fix_torchao()

import numpy as np
import pandas as pd
import torch
import transformers
import peft
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict

try:
    get_ipython  # noqa: F821
    IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False
import matplotlib
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ======================= THE SETTINGS YOU CHANGE ===========================
RUN_MODE   = "FULL_A"     # "SMOKE" (~15 min check) -> "QUICK" (seed 0) -> "FULL_A" (seeds 0-1) -> "FULL" (seeds 0-2)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"     # K-Merge's model; "unsloth/Llama-3.2-1B-Instruct" = 2nd family, no token
# ===========================================================================
RUN_MODE   = os.environ.get("KM_MODE", RUN_MODE)            # (environment overrides: only for automated testing)
MODEL_NAME = os.environ.get("KM_MODEL", MODEL_NAME)
FAKE_DATA  = os.environ.get("KM_FAKE_DATA", "0") == "1"     # offline pipeline test with generated stand-in data
OUT_DIR    = os.environ.get("KM_OUT", "/kaggle/working/kmerge_real" if os.path.isdir("/kaggle/working") else "./kmerge_real")
DTYPE_PREF = os.environ.get("KM_DTYPE", "auto")             # auto: bf16 on A100/L4/H100, fp16 + loss scaling on T4

LANGS = ["en", "de", "es", "it"]
TYPES = ["gec", "sum", "qa"]
TASKS = [f"{t}_{l}" for t in TYPES for l in LANGS]
LANG_NAME = {"en": "English", "de": "German", "es": "Spanish", "it": "Italian"}
TYPE_NAME = {"gec": "grammar correction", "sum": "summarization", "qa": "question answering"}
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 128, 0.05            # K-Merge: rank 32, alpha 128
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]  # K-Merge: query, key, value, output projections
DATA_SEED = 1234
N_GRAM = 32               # unlabeled inputs per task for RegMean's activation statistics
MAX_LEN = 512             # training sequence cap (prompt + answer, tokens)

ALL_METHODS = ["kmerge", "linear", "ties", "dare_ties", "regmean", "split", "kshare", "adaptive", "adaptive_fair",
               "wsplit", "wadaptive"]
# capacity stress test: ALL tasks in ONE slot (K = 1) that has only r_s directions (fewer than 1 per task at r_s < 12)
STRESS_METHODS = ["kmerge", "regmean", "split", "kshare", "adaptive", "wsplit", "wadaptive"]   # trimmed to fit the GPU quota
MODE = {
    "SMOKE": dict(seeds=[0], n_train=24, n_test=10, epochs=1, lr=1e-4, micro=4, accum=1, Ks=[2], orders=["random"],
                  methods=ALL_METHODS, stress_r=[8], stress_seeds=[0]),
    "QUICK": dict(seeds=[0], n_train=800, n_test=150, epochs=2, lr=1e-4, micro=4, accum=2, Ks=[2, 4],
                  orders=["random", "clustered"], methods=ALL_METHODS, stress_r=[], stress_seeds=[]),
    "FULL_A": dict(seeds=[0, 1], n_train=800, n_test=150, epochs=2, lr=1e-4, micro=4, accum=2, Ks=[2, 4],
                   orders=["random", "clustered"], methods=ALL_METHODS,          # FULL in two runs (Kaggle's 12 h limit)
                   stress_r=[16, 8, 4], stress_seeds=[0]),
    "FULL":  dict(seeds=[0, 1, 2], n_train=800, n_test=150, epochs=2, lr=1e-4, micro=4, accum=2, Ks=[2, 4],
                  orders=["random", "clustered"], methods=ALL_METHODS, stress_r=[16, 8, 4], stress_seeds=[0]),
}[RUN_MODE]
SEEDS = MODE["seeds"]
N_TRAIN, N_TEST = MODE["n_train"], MODE["n_test"]
METHOD_LIST = MODE["methods"]


# ---------------- PRE-FLIGHT CHECK ----------------
def find_local_model(name, root="/kaggle/input"):
    if os.path.isdir(name):
        return name
    if not os.path.isdir(root):
        return None
    leaf = name.split("/")[-1].lower()
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath.count(os.sep) - root.count(os.sep) >= 7:
            dirnames[:] = []
            continue
        if "config.json" in filenames and any(f.endswith((".safetensors", ".bin")) for f in filenames) \
                and leaf.replace("qwen2.5-", "") in dirpath.lower():
            return dirpath
    return None


_problems = []
if os.path.isdir("/kaggle") and not torch.cuda.is_available():
    _problems.append("No GPU is switched on.  FIX: right panel -> Settings -> Accelerator -> 'GPU T4 x2', then Run All.")
MODEL_PATH = find_local_model(MODEL_NAME) or MODEL_NAME
if not INTERNET and not FAKE_DATA:
    _problems.append("Internet is OFF: the datasets and translation models are downloaded.  FIX: Settings -> Internet -> On.")
print("PRE-FLIGHT | python", sys.version.split()[0], "| torch", torch.__version__, "| transformers",
      transformers.__version__, "| peft", peft.__version__, "| internet", "ON" if INTERNET else "OFF",
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none", "| model", MODEL_PATH)
if _problems:
    raise RuntimeError("PRE-FLIGHT FAILED:\n - " + "\n - ".join(_problems))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DTYPE_PREF == "auto":
    if DEVICE.type == "cuda":
        DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    else:
        DTYPE = torch.float32
else:
    DTYPE = {"fp32": torch.float32, "bf16": torch.bfloat16, "fp16": torch.float16}[DTYPE_PREF]
EVAL_BATCH = 24 if DEVICE.type == "cuda" else 8

MODEL_TAG = re.sub(r"[^A-Za-z0-9]+", "_", MODEL_NAME.strip("/").split("/")[-1])
CFG_TAG = f"{MODEL_TAG}_r{LORA_R}_tr{N_TRAIN}-te{N_TEST}_ep{MODE['epochs']}_lr{MODE['lr']}" + ("_FAKE" if FAKE_DATA else "")
RUN_DIR = os.path.join(OUT_DIR, CFG_TAG)
ADAPTER_DIR = os.path.join(OUT_DIR, "adapters", CFG_TAG)
DATA_DIR = os.path.join(OUT_DIR, "data")
for _d in (RUN_DIR, ADAPTER_DIR, DATA_DIR, os.path.join(DATA_DIR, "raw")):
    os.makedirs(_d, exist_ok=True)

T0 = time.time()
def log(*a):
    print(f"[{(time.time() - T0) / 60:6.1f} min]", *a, flush=True)


def restore_from_inputs(root=os.environ.get("KM_INPUT", "/kaggle/input")):
    """RESUME: copy finished work of an earlier version (Add Input -> Your Work -> this notebook). Never overwrites."""
    import shutil
    if not os.path.isdir(root):
        return
    for dirpath, dirnames, _ in os.walk(root, followlinks=True):
        if dirpath.count(os.sep) - root.count(os.sep) >= 6:
            dirnames[:] = []
            continue
        if os.path.basename(dirpath) != "kmerge_real" or os.path.abspath(dirpath) == os.path.abspath(OUT_DIR):
            continue
        dirnames[:] = []
        copied = 0
        for sdir, _, files in os.walk(dirpath, followlinks=True):     # the whole tree (every mode), so that any later
            ddir = os.path.normpath(os.path.join(OUT_DIR, os.path.relpath(sdir, dirpath)))   # version can resume from it
            for fn in sorted(files):
                s, d = os.path.join(sdir, fn), os.path.join(ddir, fn)
                if fn != "results.zip" and not os.path.exists(d):
                    os.makedirs(ddir, exist_ok=True)
                    shutil.copy2(s, d)
                    os.chmod(d, 0o644)
                    copied += 1
        if copied:
            log(f"RESUME: copied {copied} cached files from {dirpath}; finished work will be skipped")


restore_from_inputs()
log(f"RUN_MODE={RUN_MODE} | model={MODEL_NAME} | device={DEVICE} "
    f"({torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu'}) | dtype={DTYPE} | tasks={len(TASKS)}"
    + (" | FAKE DATA (offline test)" if FAKE_DATA else ""))

In [ ]:
# ============================================================================
# 2. DATA: real grammar-correction corpora + SAMSum + SQuAD, translated with OPUS-MT where needed
# ============================================================================
SOURCES = {
    "wi": "https://www.cl.cam.ac.uk/research/nl/bea2019st/data/wi+locness_v2.1.bea19.tar.gz",
    "fm": "https://github.com/adrianeboyd/boyd-wnut2018/releases/download/wnut2018/data.tar.gz",
    "cows_train": "https://huggingface.co/datasets/Ro551/COWSL2H_GEC_cleaned/resolve/main/data/train-00000-of-00001.parquet",
    "cows_test": "https://huggingface.co/datasets/Ro551/COWSL2H_GEC_cleaned/resolve/main/data/test-00000-of-00001.parquet",
    "merlin_it": "https://huggingface.co/datasets/aseifert/merlin/resolve/main/italian.jsonl",
    "samsum_train": "https://huggingface.co/datasets/knkarthick/samsum/resolve/main/train.csv",
    "samsum_test": "https://huggingface.co/datasets/knkarthick/samsum/resolve/main/test.csv",
    "squad_train": "https://huggingface.co/datasets/rajpurkar/squad/resolve/main/plain_text/train-00000-of-00001.parquet",
    "squad_val": "https://huggingface.co/datasets/rajpurkar/squad/resolve/main/plain_text/validation-00000-of-00001.parquet",
}
MT_MODELS = {"de": "Helsinki-NLP/opus-mt-en-de", "es": "Helsinki-NLP/opus-mt-en-es", "it": "Helsinki-NLP/opus-mt-en-it"}


def fetch(key):
    name = SOURCES[key].split("/")[-1]
    path = os.path.join(DATA_DIR, "raw", f"{key}__{name}")
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    for attempt in range(4):
        try:
            req = urllib.request.Request(SOURCES[key], headers={"User-Agent": "kmerge-real-bench"})
            with urllib.request.urlopen(req, timeout=180) as r, open(path + ".part", "wb") as f:
                f.write(r.read())
            os.replace(path + ".part", path)
            return path
        except Exception as e:
            log(f"download {key} failed ({e!r}), retry {attempt + 1}/4")
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"Could not download {SOURCES[key]}. Check that Internet is On.")


def detok(s, lang):
    """Undo the space-separated tokenisation of the learner corpora (applied to source and target alike)."""
    s = " ".join(str(s).split())
    s = re.sub(r" ([.,!?;:%)\]}»])", r"\1", s)
    s = re.sub(r"([(\[{«¿¡]) ", r"\1", s)
    if lang == "en":
        s = re.sub(r" (n't|'s|'re|'ve|'ll|'d|'m)\b", r"\1", s, flags=re.I)
    if lang in ("it", "fr"):
        s = re.sub(r"(\w)' (\w)", r"\1'\2", s)
    s = re.sub(r'" (.*?) "', r'"\1"', s)
    return s.strip()


def m2_pairs(text):
    """M2 (BEA-2019) -> (source, corrected) using annotator 0."""
    out = []
    for block in text.strip().split("\n\n"):
        lines = block.strip().split("\n")
        if not lines or not lines[0].startswith("S "):
            continue
        src = lines[0][2:].split()
        edits = []
        for ln in lines[1:]:
            if not ln.startswith("A "):
                continue
            f = ln[2:].split("|||")
            if len(f) < 6 or f[-1].strip() != "0" or f[1] == "noop":
                continue
            s_, e_ = map(int, f[0].split())
            if s_ < 0:
                continue
            edits.append((s_, e_, f[2]))
        tgt, off = list(src), 0
        for s_, e_, c in sorted(edits):
            c = c.split() if c.strip() else []
            tgt[s_ + off:e_ + off] = c
            off += len(c) - (e_ - s_)
        out.append((" ".join(src), " ".join(tgt)))
    return out


def _ok_gec(s, t):
    n = len(s.split())
    return 4 <= n <= 45 and 0 < len(t.split()) <= 60 and len(s) <= 320


def gec_pools(lang):
    if lang == "en":
        with tarfile.open(fetch("wi")) as tf:
            get = lambda n: tf.extractfile([m for m in tf.getnames() if m.endswith(n)][0]).read().decode("utf-8")
            pools = {"train": m2_pairs(get("ABC.train.gold.bea19.m2")), "test": m2_pairs(get("ABCN.dev.gold.bea19.m2"))}
    elif lang == "de":
        with tarfile.open(fetch("fm")) as tf:
            rd = lambda n: tf.extractfile(f"data/{n}").read().decode("utf-8").split("\n")
            pools = {sp: [(a, b) for a, b in zip(rd(f"fm-{sp}.src"), rd(f"fm-{sp}.trg")) if a.strip()]
                     for sp in ("train", "test")}
    elif lang == "es":
        pools = {}
        for sp, key in (("train", "cows_train"), ("test", "cows_test")):
            df = pd.read_parquet(fetch(key))
            pools[sp] = list(zip(df.input_text.astype(str), df.target_text.astype(str)))
    elif lang == "it":
        rows = [json.loads(l) for l in open(fetch("merlin_it"), encoding="utf-8") if l.strip()]
        pools = {"train": [], "test": []}
        for x in rows:                                   # split by learner text, so no essay is in both splits
            essay = x["id"].rsplit("-", 1)[0]
            sp = "test" if int(hashlib.md5(essay.encode()).hexdigest(), 16) % 100 < 15 else "train"
            pools[sp].append((x["original"], x["corrected"]))
    else:
        raise ValueError(lang)
    out = {}
    for sp, pairs in pools.items():
        seen, keep = set(), []
        for s, t in pairs:
            s, t = detok(s, lang), detok(t, lang)
            if _ok_gec(s, t) and s not in seen:
                seen.add(s)
                keep.append({"src": s, "tgt": t})
        out[sp] = keep
    return out


def sum_pools():
    out = {}
    for sp, key in (("train", "samsum_train"), ("test", "samsum_test")):
        df = pd.read_csv(fetch(key)).dropna(subset=["dialogue", "summary"])
        rows = []
        for d, s in zip(df.dialogue.astype(str), df.summary.astype(str)):
            d = "\n".join(l.strip() for l in d.replace("\r", "").split("\n") if l.strip())
            if 30 <= len(d.split()) <= 170 and 4 <= len(s.split()) <= 45 and d.count("\n") <= 18:
                rows.append({"src": d, "tgt": " ".join(s.split())})
        out[sp] = rows
    return out


def qa_pools():
    out = {}
    for sp, key in (("train", "squad_train"), ("test", "squad_val")):
        df = pd.read_parquet(fetch(key))
        rows = []
        for c, q, a in zip(df.context, df.question, df.answers):
            texts = list(a["text"]) if isinstance(a, dict) else list(a.get("text", []))
            if not texts:
                continue
            ans = " ".join(str(texts[0]).split())
            if len(str(c).split()) <= 150 and 1 <= len(ans.split()) <= 8:
                rows.append({"src": " ".join(str(c).split()), "q": " ".join(str(q).split()), "tgt": ans})
        out[sp] = rows
    return out


def _sample(rows, n, salt):
    rng = random.Random(DATA_SEED * 31 + int(hashlib.md5(salt.encode()).hexdigest()[:8], 16))
    rows = list(rows)
    rng.shuffle(rows)
    return rows[:n]


# ---------------- machine translation (cached; same models as K-Merge) ----------------
def _h(s):
    return hashlib.md5(s.encode("utf-8")).hexdigest()


MT_CACHE = {}


def _load_mt_cache(lang):
    if lang not in MT_CACHE:
        MT_CACHE[lang] = {}
        p = os.path.join(DATA_DIR, f"mt_{lang}.jsonl")
        if os.path.exists(p):
            for line in open(p, encoding="utf-8"):
                try:
                    d = json.loads(line)
                    MT_CACHE[lang][d["h"]] = d["t"]
                except Exception:
                    pass
    return MT_CACHE[lang]


@torch.no_grad()
def translate(texts, lang):
    cache = _load_mt_cache(lang)
    todo = sorted({t for t in texts if t and _h(t) not in cache}, key=len)
    if todo:
        from transformers import MarianMTModel, MarianTokenizer
        mt_tok = MarianTokenizer.from_pretrained(MT_MODELS[lang])
        mt = MarianMTModel.from_pretrained(MT_MODELS[lang]).to(DEVICE)
        if DEVICE.type == "cuda":
            mt = mt.half()
        mt.eval()
        t0 = time.time()
        with open(os.path.join(DATA_DIR, f"mt_{lang}.jsonl"), "a", encoding="utf-8") as f:
            bs = 48
            for i in range(0, len(todo), bs):
                chunk = todo[i:i + bs]
                enc = mt_tok(chunk, return_tensors="pt", padding=True, truncation=True, max_length=480).to(DEVICE)
                gen = mt.generate(**enc, num_beams=1, do_sample=False, max_new_tokens=512)
                for src, out in zip(chunk, mt_tok.batch_decode(gen, skip_special_tokens=True)):
                    cache[_h(src)] = out.strip()
                    f.write(json.dumps({"h": _h(src), "t": out.strip()}, ensure_ascii=False) + "\n")
        log(f"translated {len(todo)} new segments to {lang} ({time.time() - t0:.0f}s)")
        del mt
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return [cache.get(_h(t), t) if t else t for t in texts]


def _split_turn(line):
    m = re.match(r"^([^:]{1,40}):\s*(.*)$", line)
    return (m.group(1), m.group(2)) if m else ("", line)


def translate_task_rows(rows, typ, lang):
    """Translate English SAMSum / SQuAD rows: dialogue turns (speaker names kept), summaries, contexts, questions, answers."""
    if typ == "sum":
        turns = [[_split_turn(l) for l in r["src"].split("\n")] for r in rows]
        segs = [t for tt in turns for _, t in tt if re.search(r"[A-Za-z]", t)] + [r["tgt"] for r in rows]
        tr = dict(zip(segs, translate(segs, lang)))
        out = []
        for r, tt in zip(rows, turns):
            d = "\n".join((f"{sp}: " if sp else "") + tr.get(t, t) for sp, t in tt)
            out.append({"src": d, "tgt": tr.get(r["tgt"], r["tgt"])})
        return out
    segs = [r["src"] for r in rows] + [r["q"] for r in rows] + [r["tgt"] for r in rows]
    tr = dict(zip(segs, translate(segs, lang)))
    return [{"src": tr[r["src"]], "q": tr[r["q"]], "tgt": tr[r["tgt"]]} for r in rows]


# ---------------- offline stand-in data (only for testing the pipeline without Internet) ----------------
def fake_data():
    rng = random.Random(0)
    names = ["Anna", "Tom", "Lea", "Marco", "Sara", "Paul"]
    things = ["the report", "the tickets", "dinner", "the meeting", "the car", "the books"]
    data = {}
    for t in TASKS:
        typ, lang = t.split("_")
        rows = []
        for i in range(N_TRAIN + N_TEST):
            a, b, th = rng.choice(names), rng.choice(names), rng.choice(things)
            if typ == "gec":
                clean = f"{a} will bring {th} tomorrow morning ({lang})."
                src = clean.replace("will", "wil") if i % 3 else clean
                rows.append({"src": src, "tgt": clean})
            elif typ == "sum":
                rows.append({"src": f"{a}: Can you bring {th}?\n{b}: Yes, tomorrow.", "tgt": f"{b} will bring {th} tomorrow."})
            else:
                rows.append({"src": f"{a} lives in Rome. {b} lives in Paris.", "q": f"Where does {a} live?", "tgt": "Rome"})
        data[t] = {"train": rows[:N_TRAIN], "test": rows[N_TRAIN:]}
    return data


DATA_PATH = os.path.join(DATA_DIR, f"tasks_tr{N_TRAIN}-te{N_TEST}" + ("_FAKE" if FAKE_DATA else "") + ".json")
if os.path.exists(DATA_PATH):
    DATA = json.load(open(DATA_PATH, encoding="utf-8"))
    log(f"task data loaded from cache ({DATA_PATH})")
elif FAKE_DATA:
    DATA = fake_data()
    json.dump(DATA, open(DATA_PATH, "w", encoding="utf-8"), ensure_ascii=False)
else:
    DATA = {}
    for lang in LANGS:
        pools = gec_pools(lang)
        DATA[f"gec_{lang}"] = {"train": _sample(pools["train"], N_TRAIN, f"gec_{lang}_tr"),
                               "test": _sample(pools["test"], N_TEST, f"gec_{lang}_te")}
        log(f"gec_{lang}: pools train/test = {len(pools['train'])}/{len(pools['test'])}")
    for typ, fn in (("sum", sum_pools), ("qa", qa_pools)):
        pools = fn()
        base = {"train": _sample(pools["train"], N_TRAIN, f"{typ}_tr"), "test": _sample(pools["test"], N_TEST, f"{typ}_te")}
        log(f"{typ}: pools train/test = {len(pools['train'])}/{len(pools['test'])}")
        DATA[f"{typ}_en"] = base
        for lang in LANGS[1:]:
            DATA[f"{typ}_{lang}"] = {sp: translate_task_rows(base[sp], typ, lang) for sp in ("train", "test")}
    json.dump(DATA, open(DATA_PATH, "w", encoding="utf-8"), ensure_ascii=False)
for t in TASKS:
    ex = DATA[t]["test"][0]
    log(f"{t:7s}: train/test={len(DATA[t]['train'])}/{len(DATA[t]['test'])} | e.g. "
        + (ex["src"][:90].replace("\n", " / ") + (f" || Q: {ex['q'][:50]}" if "q" in ex else "") + f"  ->  {ex['tgt'][:70]}"))

In [ ]:
# ============================================================================
# 3. MODEL, LORA TRAINING, METRICS, EVALUATION
# ============================================================================
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
END_TOKEN = "<|im_end|>" if "<|im_end|>" in tok.get_vocab() else (
    "<|eot_id|>" if "<|eot_id|>" in tok.get_vocab() else tok.eos_token)
EOS_IDS = sorted({i for i in [tok.convert_tokens_to_ids(END_TOKEN), tok.eos_token_id] if i is not None and i >= 0})


def load_base(dtype):
    try:
        m = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=dtype)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=dtype)
    return m.to(DEVICE)


base_model = load_base(DTYPE)
if DTYPE == torch.float16:                                   # fp16 can overflow on some models: verify, else use fp32
    with torch.no_grad():
        _ids = tok("Hello, how are you today?", return_tensors="pt").input_ids.to(DEVICE)
        if not torch.isfinite(base_model(_ids).logits.float()).all():
            log("fp16 produced non-finite logits -> reloading the model in fp32")
            del base_model; torch.cuda.empty_cache(); DTYPE = torch.float32; base_model = load_base(DTYPE)
model = get_peft_model(base_model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                              target_modules=TARGET_MODULES, bias="none", task_type="CAUSAL_LM"))
for _n, _p in model.named_parameters():                     # adapter weights stay in fp32 (stable training, exact merging)
    if "lora_" in _n and _p.dtype != torch.float32:
        _p.data = _p.data.float()
model.eval()
AMP = DEVICE.type == "cuda" and DTYPE != torch.float32


def amp_ctx():
    import contextlib
    return torch.autocast(device_type="cuda", dtype=DTYPE) if AMP else contextlib.nullcontext()

MOD_MAP = OrderedDict((n, m) for n, m in model.named_modules()
                      if hasattr(m, "lora_A") and hasattr(m.lora_A, "keys") and "default" in m.lora_A)
N_LAYERS = 1 + max(int(re.search(r"layers\.(\d+)\.", k).group(1)) for k in MOD_MAP)
log(f"LoRA modules: {len(MOD_MAP)} ({N_LAYERS} layers x {TARGET_MODULES}) | rank {LORA_R} | AMP={AMP}")

SYSTEM = "You are a helpful assistant."


def user_msg(task, ex):
    typ, lang = task.split("_")
    L = LANG_NAME[lang]
    if typ == "gec":
        return (f"Correct the grammar and spelling of this {L} sentence. If it is already correct, repeat it unchanged. "
                f"Reply with the corrected sentence only.\n\n{ex['src']}")
    if typ == "sum":
        return f"Summarize this {L} conversation in one or two sentences, in {L}. Reply with the summary only.\n\n{ex['src']}"
    return (f"Read the {L} text and answer the question with a short phrase from the text, in {L}. "
            f"Reply with the answer only.\n\nText: {ex['src']}\n\nQuestion: {ex['q']}")


def build_prompt(task, ex):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg(task, ex)}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


PROMPTS = {t: {sp: [build_prompt(t, ex) for ex in DATA[t][sp]] for sp in ("train", "test")} for t in TASKS}


def _max_new(task):
    typ = task.split("_")[0]
    if typ == "gec":
        return min(160, max(len(tok(ex["tgt"], add_special_tokens=False).input_ids) for ex in DATA[task]["test"]) + 12)
    return 96 if typ == "sum" else 32


GEN_CFGS = {t: GenerationConfig(max_new_tokens=_max_new(t), do_sample=False, num_beams=1, repetition_penalty=1.0,
                                pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS) for t in TASKS}


def init_lora(seed):
    """PEFT's default LoRA init (kaiming-uniform A, zero B) from an explicit generator: same A-init for all tasks of a seed."""
    g = torch.Generator().manual_seed(int(seed))
    with torch.no_grad():
        for mk, m in MOD_MAP.items():
            A, B = m.lora_A["default"].weight, m.lora_B["default"].weight
            bound = 1.0 / math.sqrt(A.shape[1])
            A.copy_(((torch.rand(A.shape, generator=g) * 2 - 1) * bound).to(A.dtype))
            B.zero_()


def fac_to_model(fac):
    with torch.no_grad():
        for mk, (B, A) in fac.items():
            m = MOD_MAP[mk]
            m.lora_A["default"].weight.copy_(A.to(m.lora_A["default"].weight.dtype))
            m.lora_B["default"].weight.copy_(B.to(m.lora_B["default"].weight.dtype))


def state_to_fac(state):
    return OrderedDict((mk, (state[mk + ".lora_B.weight"].float().to(DEVICE), state[mk + ".lora_A.weight"].float().to(DEVICE)))
                       for mk in MOD_MAP)


def _tokenize_train(task):
    rows, skipped = [], 0
    for p, ex in zip(PROMPTS[task]["train"], DATA[task]["train"]):
        pi = tok(p, add_special_tokens=False).input_ids
        ti = tok(ex["tgt"] + END_TOKEN, add_special_tokens=False).input_ids
        if len(pi) + len(ti) > MAX_LEN:
            skipped += 1
            continue
        rows.append((pi + ti, [-100] * len(pi) + ti))
    return rows, skipped


def train_adapter(task, seed):
    path = os.path.join(ADAPTER_DIR, f"seed{seed}_{task}.pt")
    if os.path.exists(path):
        return torch.load(path, map_location="cpu")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    init_lora(seed)
    data, skipped = _tokenize_train(task)
    rng = random.Random(seed * 7919 + TASKS.index(task))
    params = [p for n, p in model.named_parameters() if "lora_" in n]
    for p in params:
        p.requires_grad_(True)
    opt = torch.optim.AdamW(params, lr=MODE["lr"], weight_decay=0.0)
    mb, acc = MODE["micro"], MODE["accum"]
    steps = MODE["epochs"] * math.ceil(len(data) / (mb * acc))
    warm = max(1, steps // 10)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.05, (steps - s) / max(1, steps - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP and DTYPE == torch.float16))
    model.train()
    t0, losses, k = time.time(), [], 0
    for ep in range(MODE["epochs"]):
        order = list(range(len(data))); rng.shuffle(order)
        for i in range(0, len(order), mb):
            batch = [data[j] for j in order[i:i + mb]]
            L = max(len(x[0]) for x in batch)
            ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
            lab = torch.full((len(batch), L), -100, dtype=torch.long)
            att = torch.zeros((len(batch), L), dtype=torch.long)
            for b_, (x, y) in enumerate(batch):                       # right padding for training
                ids[b_, :len(x)] = torch.tensor(x); lab[b_, :len(y)] = torch.tensor(y); att[b_, :len(x)] = 1
            with amp_ctx():
                out = model(input_ids=ids.to(DEVICE), attention_mask=att.to(DEVICE), labels=lab.to(DEVICE))
            scaler.scale(out.loss / acc).backward()
            losses.append(out.loss.item())
            k += 1
            if k % acc == 0 or i + mb >= len(order):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
    model.eval()
    state = {k_: v.detach().float().cpu().clone() for k_, v in get_peft_model_state_dict(model).items()}
    torch.save(state, path)
    log(f"trained {task} seed {seed}: {len(data)} examples ({skipped} too long), {steps} steps, "
        f"loss {np.mean(losses[:4]):.3f} -> {np.mean(losses[-4:]):.3f} ({time.time() - t0:.0f}s)")
    return state


# ---------------- metrics (one primary score per problem type, as in K-Merge) ----------------
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
ARTICLES = {"en": {"a", "an", "the"}, "de": {"der", "die", "das", "den", "dem", "des", "ein", "eine", "einen", "einem", "einer"},
            "es": {"el", "la", "los", "las", "un", "una", "unos", "unas"},
            "it": {"il", "lo", "la", "i", "gli", "le", "un", "uno", "una", "l"}}


def gec_edits(src, out):
    a, b = WORD_RE.findall(src), WORD_RE.findall(out)
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    return {(i1, i2, tuple(b[j1:j2])) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal"}


def f05(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 1.0
    r = tp / (tp + fn) if tp + fn else 1.0
    return (1.25 * p * r / (0.25 * p + r)) if (p + r) > 0 else 0.0


def _lcs(a, b):
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]


def rouge_l(pred, ref):
    a, b = WORD_RE.findall(pred.lower()), WORD_RE.findall(ref.lower())
    l = _lcs(a, b)
    if l == 0:
        return 0.0
    p, r = l / len(a), l / len(b)
    return 2 * p * r / (p + r)


def _qa_toks(s, lang):
    return [w for w in re.findall(r"\w+", s.lower()) if w not in ARTICLES.get(lang, set())]


def qa_f1(pred, gold, lang):
    p, g = _qa_toks(pred, lang), _qa_toks(gold, lang)
    if not p or not g:
        return float(p == g)
    ov = sum((Counter(p) & Counter(g)).values())
    if ov == 0:
        return 0.0
    pr, rc = ov / len(p), ov / len(g)
    return 2 * pr * rc / (pr + rc)


def score(task, preds):
    typ, lang = task.split("_")
    rows = DATA[task]["test"]
    if typ == "gec":
        tp, fp, fn = [], [], []
        for p, ex in zip(preds, rows):
            H, G = gec_edits(ex["src"], p), gec_edits(ex["src"], ex["tgt"])
            t_ = len(H & G); tp.append(t_); fp.append(len(H) - t_); fn.append(len(G) - t_)
        return {"score": f05(sum(tp), sum(fp), sum(fn)), "stats": {"tp": tp, "fp": fp, "fn": fn},
                "em": float(np.mean([" ".join(p.split()) == " ".join(ex["tgt"].split()) for p, ex in zip(preds, rows)]))}
    if typ == "sum":
        r = [rouge_l(p, ex["tgt"]) for p, ex in zip(preds, rows)]
        return {"score": float(np.mean(r)), "stats": {"v": r}}
    f = [qa_f1(p, ex["tgt"], lang) for p, ex in zip(preds, rows)]
    return {"score": float(np.mean(f)), "stats": {"v": f}}


def stat_score(task, stats, idx=None):
    """Primary score from per-example statistics; idx = array of resampled example indices (bootstrap)."""
    if task.startswith("gec"):
        tp, fp, fn = (np.asarray(stats[k]) for k in ("tp", "fp", "fn"))
        if idx is None:
            return f05(tp.sum(), fp.sum(), fn.sum())
        TP, FP, FN = tp[idx].sum(-1), fp[idx].sum(-1), fn[idx].sum(-1)
        P = np.where(TP + FP > 0, TP / np.maximum(TP + FP, 1), 1.0)
        Rr = np.where(TP + FN > 0, TP / np.maximum(TP + FN, 1), 1.0)
        return np.where(P + Rr > 0, 1.25 * P * Rr / np.maximum(0.25 * P + Rr, 1e-12), 0.0)
    v = np.asarray(stats["v"])
    return v.mean() if idx is None else v[idx].mean(-1)


# ---------------- generation + persistent evaluation cache ----------------
CACHE_PATH = os.path.join(RUN_DIR, "eval_cache.jsonl")
EVAL_CACHE = {}
if os.path.exists(CACHE_PATH):
    for line in open(CACHE_PATH, encoding="utf-8"):
        try:
            d = json.loads(line); EVAL_CACHE[d["k"]] = d["m"]
        except Exception:
            pass
PRED_PATH = os.path.join(RUN_DIR, "predictions_test.csv")


@torch.no_grad()
def generate(task):
    outs, prompts = [], PROMPTS[task]["test"]
    for i in range(0, len(prompts), EVAL_BATCH):
        enc = tok(prompts[i:i + EVAL_BATCH], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        with amp_ctx():
            gen = model.generate(**enc, generation_config=GEN_CFGS[task])
        texts = tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        outs += [next((l.strip() for l in t.strip().split("\n") if l.strip()), "") for t in texts]
    return outs


def evaluate(key, task, fac):
    """fac: adapter factors, a zero-argument function returning them (built lazily), or None for the base model."""
    ck = f"{key}|{task}"
    if ck in EVAL_CACHE:
        return EVAL_CACHE[ck]
    if fac is None:
        with model.disable_adapter():
            preds = generate(task)
    else:
        fac_to_model(fac() if callable(fac) else fac)
        preds = generate(task)
    m = score(task, preds)
    EVAL_CACHE[ck] = m
    with open(CACHE_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps({"k": ck, "m": m}) + "\n")
    pd.DataFrame([{"key": key, "task": task, "i": i, "pred": p} for i, p in enumerate(preds)]).to_csv(
        PRED_PATH, mode="a", header=not os.path.exists(PRED_PATH), index=False)
    return m


# ---------------- activation statistics for RegMean (base model, 32 unlabeled training inputs per task) ----------------
def gram_key(mk):
    return mk.replace("k_proj", "q_proj").replace("v_proj", "q_proj")     # q, k and v read the same input


@torch.no_grad()
def collect_grams(task):
    keys = sorted({gram_key(mk) for mk in MOD_MAP})
    grams = {k: None for k in keys}
    counts = {k: 0 for k in keys}
    state = {"mask": None}
    hooks = []
    for k in keys:
        def hook(mod, args, k=k):
            x = args[0]
            with torch.autocast(device_type=x.device.type, enabled=False):   # the forward runs under fp16 autocast:
                xv = x[state["mask"]].float()                                 # without this guard the Gram matmul is
                g = xv.T @ xv                                                 # done in fp16 and overflows to inf
            grams[k] = g if grams[k] is None else grams[k] + g
            counts[k] += xv.shape[0]
        hooks.append(MOD_MAP[k].register_forward_pre_hook(hook))
    prompts = PROMPTS[task]["train"][:N_GRAM]
    try:
        with model.disable_adapter():
            for i in range(0, len(prompts), 8):
                enc = tok(prompts[i:i + 8], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
                state["mask"] = enc["attention_mask"].bool()
                with amp_ctx():
                    model(**enc)
    finally:
        for h in hooks:
            h.remove()
    out, bad, low, high = {}, 0, 1.0, 0.0
    for k in keys:
        G = grams[k] / max(1, counts[k])
        bad += int((~torch.isfinite(G)).sum())
        d = torch.diagonal(G)
        low = min(low, float(d.min() / d.mean().clamp_min(1e-30)))
        high = max(high, float(d.max()))
        out[k] = G.to(torch.bfloat16).cpu()   # bf16 keeps fp32's range: fp16 overflowed/underflowed on Qwen2.5 channels
    GRAM_HEALTH[task] = {"nonfinite": bad, "min_rel_channel": low, "max_diag": high}
    return out


GRAM_HEALTH = {}


def prep_gram(C, beta=0.9):
    """RegMean's off-diagonal shrinkage, then normalised to mean eigenvalue 1 (equal task weight)."""
    C = (C + C.T) / 2
    C = beta * C + (1 - beta) * torch.diag(torch.diag(C))
    return C / (torch.trace(C) / C.shape[0]).clamp_min(1e-12)

In [ ]:
# ============================================================================
# 4. HOW A SLOT STORES SEVERAL TASKS: merging (one adapter) vs sharing (task-conditional), same storage
#    adapter = {module: (B [d_out x r], A [r x d_in])}, delta W = B @ A
# ============================================================================
R = LORA_R


def lr_svd(L, Rt):
    """Exact SVD of L @ Rt without forming it."""
    Ql, Rl = torch.linalg.qr(L)
    Qr, Rr = torch.linalg.qr(Rt.T)
    U, S, Vh = torch.linalg.svd(Rl @ Rr.T, full_matrices=False)
    return Ql @ U, S, Qr @ Vh.T


@contextlib.contextmanager
def slot_rank(r):
    """Temporarily give every slot r directions instead of LORA_R (capacity stress test)."""
    global R
    old, R = R, int(r)
    try:
        yield
    finally:
        R = old


def _pad_rank(B, A, r=None):
    r = R if r is None else r
    k = B.shape[1]
    if k < r:
        B = torch.cat([B, B.new_zeros(B.shape[0], r - k)], 1)
        A = torch.cat([A, A.new_zeros(r - k, A.shape[1])], 0)
    return B, A


def _to_rank(U, S, V, r=None):
    r = R if r is None else r
    k = min(r, S.numel())
    s = S[:k].clamp_min(0).sqrt()
    return _pad_rank(U[:, :k] * s, s[:, None] * V[:, :k].T, r)


def trunc_lr(L, Rt, r=None):
    U, S, V = lr_svd(L, Rt)
    return _to_rank(U, S, V, r)


def trunc_dense(M, r=None):
    r = R if r is None else r
    torch.manual_seed(0)
    U, S, V = torch.svd_lowrank(M, q=min(r + 10, min(M.shape)), niter=4)
    return _to_rank(U, S, V, r)


def ip(a, b):
    return ((a[0].T @ b[0]) * (a[1] @ b[1].T)).sum()


def _energy(a):
    return float(ip(a, a))


# ---------- merging: one adapter for every task in the slot ----------
def m_kmerge(ms, ctx):
    """K-Merge Eq. 8 (history-weighted running average of the LoRA matrices A and B; equal weights after all arrivals).
    If the slot has fewer directions than the adapters (stress test), the average is truncated to the slot's rank."""
    B, A = torch.stack([m[0] for m in ms]).mean(0), torch.stack([m[1] for m in ms]).mean(0)
    return [(B, A) if B.shape[1] <= R else trunc_lr(B, A)], []


def m_linear(ms, ctx):
    n = len(ms)
    return [trunc_lr(torch.cat([m[0] / n for m in ms], 1), torch.cat([m[1] for m in ms], 0))], []


def _trim(T, keep):
    k = max(1, int(round(keep * T.numel())))
    if k >= T.numel():
        return T
    thr = T.abs().flatten().kthvalue(T.numel() - k + 1).values
    return T * (T.abs() >= thr)


def _sign_mean(Ts):
    el = torch.sign(Ts.sum(0))
    mask = ((torch.sign(Ts) == el) & (Ts != 0)).float()
    return (Ts * mask).sum(0) / mask.sum(0).clamp_min(1)


def m_ties(ms, ctx, keep=0.2):
    return [trunc_dense(_sign_mean(torch.stack([_trim(m[0] @ m[1], keep) for m in ms])))], []


def m_dare_ties(ms, ctx, density=0.5):
    g = torch.Generator(device=ms[0][0].device).manual_seed(int(ctx["seed"]))
    Ts = []
    for m in ms:
        T = m[0] @ m[1]
        Ts.append(T * (torch.rand(T.shape, generator=g, device=T.device) < density).float() / density)
    return [trunc_dense(_sign_mean(torch.stack(Ts)))], []


REGMEAN_FALLBACKS = [0]


def m_regmean(ms, ctx):
    """RegMean: argmin_X sum_t ||X x - dW_t x||^2 over each task's real inputs = (sum dW_t C_t)(sum C_t)^-1, then rank r.
    A tiny ridge (1e-4 x mean eigenvalue) keeps the solve defined when some input channel is (almost) never active."""
    Cs = [prep_gram(torch.nan_to_num(G.float().to(ms[0][0].device))) for G in ctx["grams"]]
    N = sum(B @ (A @ C) for (B, A), C in zip(ms, Cs))
    C = sum(Cs)
    C = C + 1e-4 * (torch.trace(C) / C.shape[0]).clamp_min(1e-12) * torch.eye(C.shape[0], device=C.device, dtype=C.dtype)
    try:
        X = torch.linalg.solve(C, N.T).T
    except RuntimeError:                                   # torch.linalg.LinAlgError is a RuntimeError
        REGMEAN_FALLBACKS[0] += 1
        X = N @ torch.linalg.pinv(C, hermitian=True)
    return [trunc_dense(X)], []


# ---------- sharing: the slot keeps a small private part of every task (task identity is known at inference) ----------
def s_split(ms, ctx):
    """Each task keeps its own top singular directions; the rank budget is divided equally (the TSV-C idea)."""
    M = len(ms)
    ks = [R // M + (1 if i < R % M else 0) for i in range(M)]
    outs, kept = [], []
    for (B, A), k in zip(ms, ks):
        U, S, V = lr_svd(B, A)
        k = min(k, S.numel())
        outs.append(_pad_rank(U[:, :k] * S[:k], V[:, :k].T))
        kept.append(float((S[:k] ** 2).sum() / (S ** 2).sum().clamp_min(1e-30)))
    return outs, kept


def _top_left(M, k):
    return torch.linalg.svd(M, full_matrices=False)[0][:, :k]


def _psd_sqrt(M):
    e, Q = torch.linalg.eigh((M + M.T) / 2)
    return (Q * e.clamp_min(0).sqrt()) @ Q.T


def s_kshare(ms, ctx, iters=6):
    """K-Share: one shared basis (U, V, the size of one LoRA) + a small r x r core per task, energy-balanced HOOI."""
    s = min(R, ms[0][0].shape[0], ms[0][1].shape[1])
    en = [_energy(m) for m in ms]
    c = [1.0 / max(e_, 1e-30) for e_ in en]
    U = _top_left(torch.cat([ci ** 0.5 * B @ _psd_sqrt(A @ A.T) for ci, (B, A) in zip(c, ms)], 1), s)
    V = _top_left(torch.cat([ci ** 0.5 * A.T @ _psd_sqrt(B.T @ B) for ci, (B, A) in zip(c, ms)], 1), s)
    for _ in range(iters):
        U = _top_left(torch.cat([ci ** 0.5 * B @ (A @ V) for ci, (B, A) in zip(c, ms)], 1), s)
        V = _top_left(torch.cat([ci ** 0.5 * A.T @ (B.T @ U) for ci, (B, A) in zip(c, ms)], 1), s)
    outs, kept = [], []
    for (B, A), e_ in zip(ms, en):
        G = (U.T @ B) @ (A @ V)
        outs.append(_pad_rank(U @ G, V.T))
        kept.append(float((G ** 2).sum()) / max(e_, 1e-30))
    return outs, kept


def alloc_sum(rels, budget):
    """Give each direction of the slot's rank budget to the task where it keeps the most of its own energy
    (greedy = optimal for this separable objective); every task gets at least one direction."""
    M = len(rels)
    k = [1 if budget >= M else 0 for _ in range(M)]
    budget -= sum(k)
    heap = [(-float(rels[t][k[t]]), t) for t in range(M) if k[t] < len(rels[t])]
    heapq.heapify(heap)
    while budget > 0 and heap:
        _, t = heapq.heappop(heap)
        k[t] += 1; budget -= 1
        if k[t] < len(rels[t]):
            heapq.heappush(heap, (-float(rels[t][k[t]]), t))
    return k


def alloc_fair(rels, budget):
    """Max-min version: the next direction always goes to the task that currently keeps the smallest share of its energy."""
    M = len(rels)
    k = [1 if budget >= M else 0 for _ in range(M)]
    kept = [float(rels[t][:k[t]].sum()) for t in range(M)]
    budget -= sum(k)
    while budget > 0:
        cand = [t for t in range(M) if k[t] < len(rels[t])]
        if not cand:
            break
        t = min(cand, key=lambda t_: kept[t_])
        kept[t] += float(rels[t][k[t]]); k[t] += 1; budget -= 1
    return k


def _s_adaptive(ms, fair):
    svds = [lr_svd(B, A) for B, A in ms]
    rels = [(S ** 2) / (S ** 2).sum().clamp_min(1e-30) for _, S, _ in svds]
    ks = (alloc_fair if fair else alloc_sum)(rels, R)
    outs, kept = [], []
    for (U, S, V), k, rel in zip(svds, ks, rels):
        outs.append(_pad_rank(U[:, :k] * S[:k], V[:, :k].T))
        kept.append(float(rel[:k].sum()))
    return outs, kept


def s_adaptive(ms, ctx):
    """NEW (adaptive slots): each task gets as many of the slot's r directions as it needs, decided per layer and module
    from how fast its singular values decay. Storage = exactly one LoRA (sum of directions = r)."""
    return _s_adaptive(ms, fair=False)


def s_adaptive_fair(ms, ctx):
    """NEW (adaptive slots, fair): same budget, allocated to protect the task that would lose the most."""
    return _s_adaptive(ms, fair=True)


# ---------- NEW: activation-aware sharing (each task keeps what matters on its own real inputs) ----------
WHITEN_FALLBACKS = [0]


def _chol_factor(G, dev):
    """Lower-triangular L with L L^T = C (the task's shrunk, normalised input Gram, plus a tiny ridge)."""
    C = prep_gram(torch.nan_to_num(G.float().to(dev)))
    d = C.shape[0]
    eps = 1e-4 * float((torch.trace(C) / d).clamp_min(1e-12))
    I = torch.eye(d, device=dev, dtype=C.dtype)
    for _ in range(6):
        L, info = torch.linalg.cholesky_ex(C + eps * I)
        if int(info) == 0:
            return L
        eps *= 10; WHITEN_FALLBACKS[0] += 1
    return torch.diagonal(C).clamp_min(eps).sqrt().diag()          # last resort: per-channel whitening


def _s_whitened(ms, ctx, adaptive):
    """Truncate each task in its input-whitened space. ||(X - dW) L||_F is the error on the task's own inputs, so the best
    rank-k X is [dW L]_k L^-1 (Eckart-Young after whitening). Storage as split: the task ranks add up to r."""
    Ls = [_chol_factor(G, ms[0][0].device) for G in ctx["grams"]]
    svds = [lr_svd(B, A @ L) for (B, A), L in zip(ms, Ls)]
    rels = [(S ** 2) / (S ** 2).sum().clamp_min(1e-30) for _, S, _ in svds]
    M = len(ms)
    ks = alloc_sum(rels, R) if adaptive else [R // M + (1 if i < R % M else 0) for i in range(M)]
    outs, kept = [], []
    for (U, S, V), k, rel, L in zip(svds, ks, rels, Ls):
        k = min(k, S.numel())
        A_k = torch.linalg.solve_triangular(L.T, V[:, :k], upper=True).T          # V_k^T L^-1
        outs.append(_pad_rank(U[:, :k] * S[:k], A_k))
        kept.append(float(rel[:k].sum()))
    return outs, kept


def s_wsplit(ms, ctx):
    """NEW (activation-aware split): equal ranks, but each task keeps the directions that change its outputs most on its
    own real inputs (32 unlabeled examples, the data RegMean uses) instead of its largest weight directions."""
    return _s_whitened(ms, ctx, adaptive=False)


def s_wadaptive(ms, ctx):
    """NEW (activation-aware adaptive): the slot's r directions go, per layer and module, to whichever task would lose the
    most output quality on its own inputs without them. Storage = exactly one LoRA."""
    return _s_whitened(ms, ctx, adaptive=True)


METHODS = OrderedDict([
    ("kmerge",        dict(fn=m_kmerge,        kind="merge", group="merging (K-Merge)",  grams=False, storage="1 LoRA")),
    ("linear",        dict(fn=m_linear,        kind="merge", group="merging (prior)",    grams=False, storage="1 LoRA")),
    ("ties",          dict(fn=m_ties,          kind="merge", group="merging (prior)",    grams=False, storage="1 LoRA")),
    ("dare_ties",     dict(fn=m_dare_ties,     kind="merge", group="merging (prior)",    grams=False, storage="1 LoRA")),
    ("regmean",       dict(fn=m_regmean,       kind="merge", group="merging (prior)",    grams=True,  storage="1 LoRA (+32 inputs per task)")),
    ("split",         dict(fn=s_split,         kind="share", group="sharing (prior idea)", grams=False, storage="1 LoRA")),
    ("kshare",        dict(fn=s_kshare,        kind="share", group="sharing (new)",      grams=False, storage="1 LoRA + r x r per task")),
    ("adaptive",      dict(fn=s_adaptive,      kind="share", group="sharing (new)",      grams=False, storage="1 LoRA")),
    ("adaptive_fair", dict(fn=s_adaptive_fair, kind="share", group="sharing (new)",      grams=False, storage="1 LoRA")),
    ("wsplit",        dict(fn=s_wsplit,        kind="share", group="sharing (new)",      grams=True,  storage="1 LoRA (+32 inputs per task)")),
    ("wadaptive",     dict(fn=s_wadaptive,     kind="share", group="sharing (new)",      grams=True,  storage="1 LoRA (+32 inputs per task)")),
])


def build_slot(method, slot, seed):
    """Representation of one slot holding the tasks in `slot`: list of per-task adapters (identical for merging)."""
    spec = METHODS[method]
    facs = [FAC[seed][t] for t in slot]
    outs = [OrderedDict() for _ in slot]
    kept = []
    for i, mk in enumerate(facs[0]):
        ctx = {"seed": seed * 1000003 + i}
        if spec["grams"]:
            ctx["grams"] = [GRAMS[t][gram_key(mk)] for t in slot]
        reps, kk = spec["fn"]([f[mk] for f in facs], ctx)
        for j in range(len(slot)):
            B_, A_ = reps[0] if len(reps) == 1 else reps[j]
            outs[j][mk] = _pad_rank(B_, A_, LORA_R)        # zero columns: the model's LoRA modules have LORA_R of them
        kept += kk
    return outs, (float(np.mean(kept)) if kept else float("nan"))


def kmerge_cos(fa, fb):
    """K-Merge's similarity: layer-averaged cosine of the flattened updates."""
    return float(np.mean([float(ip(fa[mk], fb[mk]) / (ip(fa[mk], fa[mk]).sqrt() * ip(fb[mk], fb[mk]).sqrt()).clamp_min(1e-30))
                          for mk in fa]))

In [ ]:
# ============================================================================
# 5. SELF-TEST OF THE SLOT MATH (seconds; must pass before any experiment)
# ============================================================================
def slot_selftest():
    g = torch.Generator().manual_seed(0)
    d_out, d_in = 48, 40

    def rnd(decay):
        B = torch.randn(d_out, R, generator=g) * (decay ** torch.arange(R).float())[None, :]
        return B.to(DEVICE), (torch.randn(R, d_in, generator=g) / math.sqrt(d_in)).to(DEVICE)

    a, b, c = rnd(0.6), rnd(0.97), rnd(0.8)            # fast-, slow- and medium-decaying spectra
    dense = lambda x: x[0] @ x[1]
    rel = lambda x, y: float((dense(x) - dense(y)).norm() / dense(y).norm())
    checks = []
    for name in ("split", "kshare", "adaptive", "adaptive_fair"):
        out, _ = METHODS[name]["fn"]([a], {"seed": 0})
        checks.append((f"{name}: a slot with one task returns that task", rel(out[0], a) < 1e-4))
    for name in ("kmerge", "linear"):
        out, _ = METHODS[name]["fn"]([a, a], {"seed": 0})
        checks.append((f"{name}: merging a task with itself returns it", rel(out[0], a) < 1e-4))
    C = torch.randn(d_in, 3 * d_in, generator=g); C = (C @ C.T / (3 * d_in)).to(DEVICE)
    out, _ = m_regmean([a], {"grams": [C.cpu().half()]})
    checks.append(("regmean: one task is returned unchanged", rel(out[0], a) < 2e-2))
    rels = [(lr_svd(*x)[1] ** 2) / (lr_svd(*x)[1] ** 2).sum() for x in (a, b, c)]
    for fn in (alloc_sum, alloc_fair):
        k = fn(rels, R)
        checks.append((f"{fn.__name__}: uses exactly r directions, every task >= 1", sum(k) == R and min(k) >= 1))
    ks = alloc_sum(rels, R)
    checks.append(("adaptive gives the slow-decaying task more directions than the fast one", ks[1] > ks[0]))
    kept = lambda k: sum(float(r_[:k_].sum()) for r_, k_ in zip(rels, k))
    unif = [R // 3 + (1 if i < R % 3 else 0) for i in range(3)]
    checks.append(("adaptive keeps at least as much total relative energy as the equal split", kept(ks) >= kept(unif) - 1e-9))
    kf = alloc_fair(rels, R)
    mn = lambda k: min(float(r_[:k_].sum()) for r_, k_ in zip(rels, k))
    checks.append(("adaptive_fair keeps at least the equal split's worst-task share", mn(kf) >= mn(unif) - 1e-9))
    for name in ("split", "kshare", "adaptive", "adaptive_fair", "linear", "ties", "dare_ties"):
        out, _ = METHODS[name]["fn"]([a, b, c], {"seed": 1})
        checks.append((f"{name}: rank-{R} factors for 3 tasks", all(o[0].shape == (d_out, R) and o[1].shape == (R, d_in) for o in out)))
    # activation-aware sharing: exact for one task, and never worse than split on the task's own inputs
    Xa = torch.randn(4 * d_in, d_in, generator=g) * (0.85 ** torch.arange(d_in).float())[None, :]   # anisotropic inputs
    Ca = (Xa.T @ Xa / (4 * d_in)).to(DEVICE)
    for name in ("wsplit", "wadaptive"):
        out, _ = METHODS[name]["fn"]([a], {"seed": 0, "grams": [Ca.cpu()]})
        checks.append((f"{name}: a slot with one task returns that task", rel(out[0], a) < 1e-3))
    ws, _ = s_wsplit([a, b, c], {"seed": 1, "grams": [Ca.cpu()] * 3})
    sp, _ = s_split([a, b, c], {"seed": 1})
    Cp = prep_gram(Ca.float())
    werr = lambda x, y: float(torch.trace((dense(x) - dense(y)) @ Cp @ (dense(x) - dense(y)).T))
    checks.append(("wsplit loses less than split on each task's own inputs",
                   all(werr(o, t) <= werr(s_, t) * (1 + 1e-3) + 1e-9 for o, s_, t in zip(ws, sp, (a, b, c)))))
    for name in ("wsplit", "wadaptive"):
        out, _ = METHODS[name]["fn"]([a, b, c], {"seed": 1, "grams": [Ca.cpu()] * 3})
        checks.append((f"{name}: rank-{R} factors for 3 tasks", all(o[0].shape == (d_out, R) and o[1].shape == (R, d_in) for o in out)))
    # capacity stress test: a slot with only 8 directions really stores at most 8 (K-Share: 8 shared + small cores)
    with slot_rank(8):
        for name in STRESS_METHODS:
            ctx_ = {"seed": 2, "grams": [Ca.cpu()] * 3} if METHODS[name]["grams"] else {"seed": 2}
            out, _ = METHODS[name]["fn"]([a, b, c], ctx_)
            rk = [int(torch.linalg.matrix_rank(dense(o).float().cpu(), rtol=1e-4)) for o in out]
            ok = sum(rk) <= 8 if (METHODS[name]["kind"] == "share" and name != "kshare") else max(rk) <= 8
            checks.append((f"stress: {name} keeps at most 8 directions in a rank-8 slot", ok))
    bad = [n for n, ok in checks if not ok]
    log(f"SLOT MATH SELF-TEST: {len(checks) - len(bad)}/{len(checks)} checks passed")
    fatal = [n for n in bad if not n.startswith(("wsplit", "wadaptive", "stress"))]
    if fatal:
        raise AssertionError("self-test failed: " + "; ".join(bad))
    if any(n.startswith(("wsplit", "wadaptive")) for n in bad):
        for m_ in ("wsplit", "wadaptive"):
            if m_ in METHOD_LIST:
                METHOD_LIST.remove(m_)
        log("WARNING: activation-aware sharing failed its self-test and is SKIPPED in this run: " + "; ".join(bad))
    if any(n.startswith("stress") for n in bad):
        MODE["stress_r"] = []
        log("WARNING: the capacity stress test failed its self-test and is SKIPPED in this run: " + "; ".join(bad))


slot_selftest()

In [ ]:
# ============================================================================
# 6. ADAPTERS: one LoRA per task and seed, plus the base model and single-task scores (the normalisers)
# ============================================================================
GRAMS = {}                      # RegMean's input statistics need only the base model: collected first, so a problem shows in minutes
if any(METHODS[m]["grams"] for m in METHOD_LIST):
    t0 = time.time()
    for t in TASKS:
        GRAMS[t] = collect_grams(t)
    _h = GRAM_HEALTH.values()
    log(f"RegMean input statistics ready for {len(TASKS)} tasks ({time.time() - t0:.0f}s, kept on CPU, bf16) | "
        f"non-finite entries {sum(h['nonfinite'] for h in _h)} | weakest channel / mean channel energy "
        f"{min(h['min_rel_channel'] for h in _h):.2e} | largest channel energy {max(h['max_diag'] for h in _h):.3g}")
    if sum(h["nonfinite"] for h in _h):
        raise RuntimeError("RegMean input statistics contain inf/NaN (activations overflowed). "
                           "FIX: set KM_DTYPE=fp32 (or DTYPE_PREF='fp32') and run again.")

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
_STATES = {(seed, t): train_adapter(t, seed) for seed in SEEDS for t in TASKS}   # train everything first (CPU copies),
FAC = {seed: {t: state_to_fac(_STATES[(seed, t)]) for t in TASKS} for seed in SEEDS}   # then move factors to the GPU
del _STATES
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

REF_ROWS = []
for t in TASKS:
    mb_ = evaluate("base", t, None)
    for seed in SEEDS:
        ms_ = evaluate(f"solo|s{seed}", t, FAC[seed][t])
        REF_ROWS.append({"task": t, "type": t.split("_")[0], "lang": t.split("_")[1], "seed": seed, "base": mb_["score"],
                         "solo": ms_["score"], "gain": ms_["score"] - mb_["score"]})
REF = pd.DataFrame(REF_ROWS)
REF.to_csv(os.path.join(RUN_DIR, "adapter_quality.csv"), index=False)
print(REF.groupby("task")[["base", "solo", "gain"]].mean().loc[TASKS].round(3).to_string())
for _r in REF.itertuples():
    if _r.gain < 0.01:
        log(f"NOTE {_r.task} seed {_r.seed}: the adapter barely beats the base model (gain {_r.gain:+.3f})")

In [ ]:
# ============================================================================
# 7. ONLINE K-MERGE STREAM: tasks arrive one at a time, K slots, K-Merge's cosine rule decides which tasks share a slot.
#    Every method gets the SAME slots; only the way a slot stores its tasks differs. Then every task is evaluated.
# ============================================================================
def make_orders(seed):
    rng = random.Random(1000 + seed)
    rand = TASKS[:]; rng.shuffle(rand)
    types = TYPES[:]; rng.shuffle(types)
    clustered = []                                         # K-Merge's worst case: same problem type arrives in a row
    for ty in types:
        ls = LANGS[:]; rng.shuffle(ls)
        clustered += [f"{ty}_{l}" for l in ls]
    return {"random": rand, "clustered": clustered}


def kmerge_partition(seed, order, K):
    slots = []
    for t in order:
        f = FAC[seed][t]
        if len(slots) < K:
            slots.append([[t], f]); continue
        sims = [kmerge_cos(f, s[1]) for s in slots]
        j = int(np.argmax(sims)); n = len(slots[j][0])
        slots[j][1] = OrderedDict((mk, ((n * slots[j][1][mk][0] + f[mk][0]) / (n + 1), (n * slots[j][1][mk][1] + f[mk][1]) / (n + 1)))
                                  for mk in f)
        slots[j][0].append(t)
    return [sorted(s[0], key=TASKS.index) for s in slots]


STREAM_CSV = os.path.join(RUN_DIR, "stream_results.csv")
PART_CSV = os.path.join(RUN_DIR, "partitions.csv")
DONE = set()
if os.path.exists(STREAM_CSV):
    _d = pd.read_csv(STREAM_CSV)
    DONE = set(zip(_d.seed, _d.K, _d.order, _d.method))
SOLO = {(r.seed, r.task): r.solo for r in REF.itertuples()}
BASE = {r.task: r.base for r in REF.itertuples()}


def nz(x, s):
    return x / s if s >= 0.05 else x


part_rows, FAILED = [], []
for seed in SEEDS:
    orders = make_orders(seed)
    for K in MODE["Ks"]:
        for oname in MODE["orders"]:
            part = kmerge_partition(seed, orders[oname], K)
            part_rows += [{"seed": seed, "K": K, "order": oname, "slot": i, "tasks": "+".join(s), "size": len(s)}
                          for i, s in enumerate(part)]
            log(f"seed {seed} K={K} {oname:9s}: slots = " + " | ".join("+".join(s) for s in part))
            for method in METHOD_LIST:
                if (seed, K, oname, method) in DONE:
                    continue
                t0, rows = time.time(), []
                try:                                       # one failing method must not stop a many-hour run
                    for slot in part:
                        skey = "+".join(slot)
                        holder = {}

                        def get(j, slot=slot, holder=holder):
                            if "reps" not in holder:
                                holder["reps"], holder["kept"] = build_slot(method, slot, seed)
                            return holder["reps"][j]
                        for j, t in enumerate(slot):
                            m = evaluate(f"{method}|s{seed}|{skey}", t, (lambda j=j: get(j)))
                            rows.append({"seed": seed, "K": K, "order": oname, "method": method, "kind": METHODS[method]["kind"],
                                         "task": t, "type": t.split("_")[0], "lang": t.split("_")[1], "slot": skey,
                                         "size": len(slot), "score": m["score"], "solo": SOLO[(seed, t)], "base": BASE[t],
                                         "nscore": nz(m["score"], SOLO[(seed, t)]), "kept": holder.get("kept", np.nan)})
                except Exception as ex:
                    FAILED.append(f"seed {seed} K={K} {oname} {method}: {ex!r}"[:300])
                    log(f"   {method:13s} FAILED and skipped: {ex!r}"[:400])
                    if DEVICE.type == "cuda":
                        torch.cuda.empty_cache()
                    continue
                df_ = pd.DataFrame(rows)
                df_.to_csv(STREAM_CSV, mode="a", header=not os.path.exists(STREAM_CSV), index=False)
                log(f"   {method:13s} mean normalised score {df_.nscore.mean():.3f} | worst task {df_.nscore.min():.3f} "
                    f"| {' '.join(f'{ty} {df_[df_.type == ty].nscore.mean():.3f}' for ty in TYPES)} ({time.time() - t0:.0f}s)")
pd.DataFrame(part_rows).to_csv(PART_CSV, index=False)
log("online stream benchmark done")

# ---------------- capacity stress test: ALL tasks in ONE slot (K = 1) that has only r_s directions ----------------
STRESS_CSV = os.path.join(RUN_DIR, "stress_results.csv")
SDONE = set()
if os.path.exists(STRESS_CSV):
    _d = pd.read_csv(STRESS_CSV)
    SDONE = set(zip(_d.seed, _d.r_slot, _d.method))
for seed in MODE.get("stress_seeds", []):
    slot_all, skey_all = TASKS[:], "+".join(TASKS)
    for rs in MODE.get("stress_r", []):
        for method in [m for m in STRESS_METHODS if m in METHOD_LIST]:
            if (seed, rs, method) in SDONE:
                continue
            t0, rows, holder = time.time(), [], {}
            try:
                with slot_rank(rs):
                    def get(j, holder=holder, method=method, seed=seed):
                        if "reps" not in holder:
                            holder["reps"], holder["kept"] = build_slot(method, slot_all, seed)
                        return holder["reps"][j]
                    for j, t in enumerate(slot_all):
                        m = evaluate(f"{method}|s{seed}|r{rs}|{skey_all}", t, (lambda j=j: get(j)))
                        rows.append({"seed": seed, "r_slot": rs, "method": method, "kind": METHODS[method]["kind"],
                                     "task": t, "type": t.split("_")[0], "lang": t.split("_")[1], "score": m["score"],
                                     "solo": SOLO[(seed, t)], "base": BASE[t], "nscore": nz(m["score"], SOLO[(seed, t)]),
                                     "kept": holder.get("kept", np.nan)})
            except Exception as ex:
                FAILED.append(f"stress seed {seed} r_s={rs} {method}: {ex!r}"[:300])
                log(f"   stress r_s={rs:2d} {method:13s} FAILED and skipped: {ex!r}"[:400])
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
                continue
            df_ = pd.DataFrame(rows)
            df_.to_csv(STRESS_CSV, mode="a", header=not os.path.exists(STRESS_CSV), index=False)
            log(f"   stress: 1 slot x {rs:2d} directions for {len(TASKS)} tasks | {method:13s} mean normalised score "
                f"{df_.nscore.mean():.3f} | worst task {df_.nscore.min():.3f} ({time.time() - t0:.0f}s)")
if MODE.get("stress_r"):
    log("capacity stress test done")

In [ ]:
# ============================================================================
# 8. ANALYSIS: leaderboard per K, significance (paired bootstrap over test examples), figures, summary
# ============================================================================
st = pd.read_csv(STREAM_CSV)
st = st[st.seed.isin(SEEDS) & st.method.isin(METHOD_LIST) & st.K.isin(MODE["Ks"]) & st.order.isin(MODE["orders"])]
st = st.drop_duplicates(["seed", "K", "order", "method", "task"], keep="last")


def board_for(K):
    d = st[st.K == K]
    b = d.groupby("method").nscore.mean().to_frame("mean")
    b = b.join(d.pivot_table(index="method", columns="type", values="nscore", aggfunc="mean").reindex(columns=TYPES))
    b["worst_task"] = d.groupby(["method", "seed", "order"]).nscore.min().groupby("method").mean()
    b["group"] = [METHODS[m]["group"] for m in b.index]
    b["storage"] = [METHODS[m]["storage"] for m in b.index]
    return b.sort_values("mean", ascending=False)


BOARDS = {K: board_for(K) for K in MODE["Ks"]}


# ---- paired bootstrap: every method is re-scored on the same resampled test examples ----
def bootstrap(K, B=2000, rs=0):
    d = st[st.K == K]
    rng = np.random.default_rng(rs)
    idx = {t: rng.integers(0, len(DATA[t]["test"]), (B, len(DATA[t]["test"]))) for t in TASKS}
    solo_b = {(s, t): stat_score(t, EVAL_CACHE[f"solo|s{s}|{t}"]["stats"], idx[t]) for s in SEEDS for t in TASKS}
    per = defaultdict(list)                      # method -> list of (B,) normalised scores, one per (seed, order, task)
    worst = defaultdict(lambda: defaultdict(list))
    for r in d.itertuples():
        m = EVAL_CACHE.get(f"{r.method}|s{r.seed}|{r.slot}|{r.task}")
        if m is None:
            continue
        sb, so = stat_score(r.task, m["stats"], idx[r.task]), solo_b[(r.seed, r.task)]
        v = np.where(so >= 0.05, sb / np.maximum(so, 1e-9), sb)
        per[r.method].append(v)
        worst[r.method][(r.seed, r.order)].append(v)
    mean_b = {m: np.mean(v, axis=0) for m, v in per.items()}
    worst_b = {m: np.mean([np.min(np.stack(g), axis=0) for g in w.values()], axis=0) for m, w in worst.items()}
    return mean_b, worst_b


def pval(x):
    return float(min(1.0, 2 * min((x <= 0).mean(), (x >= 0).mean())))


def holm(ps):
    ps = np.asarray(ps, float); o = np.argsort(ps); adj = np.empty(len(ps)); run = 0.0
    for rank, i in enumerate(o):
        run = max(run, (len(ps) - rank) * ps[i]); adj[i] = min(1.0, run)
    return adj


SIG, CON = {}, {}
for K in MODE["Ks"]:
    mb_, wb_ = bootstrap(K)
    rows = []
    for m in BOARDS[K].index:
        if m not in mb_:
            continue
        lo, hi = np.percentile(mb_[m], [2.5, 97.5])
        row = {"method": m, "mean": BOARDS[K].loc[m, "mean"], "95CI": f"[{lo:.3f}, {hi:.3f}]",
               "worst_task": BOARDS[K].loc[m, "worst_task"]}
        if m != "kmerge" and "kmerge" in mb_:
            row["d_vs_kmerge"] = BOARDS[K].loc[m, "mean"] - BOARDS[K].loc["kmerge", "mean"]
            row["p_vs_kmerge"] = pval(mb_[m] - mb_["kmerge"])
        rows.append(row)
    sg = pd.DataFrame(rows)
    if "p_vs_kmerge" in sg:
        msk = sg.p_vs_kmerge.notna()
        sg.loc[msk, "p_holm"] = holm(sg.loc[msk, "p_vs_kmerge"].values)
    SIG[K] = sg
    best_merge = max([m for m in mb_ if METHODS[m]["kind"] == "merge"], key=lambda m: BOARDS[K].loc[m, "mean"], default=None)
    cons = []
    for a_, b_ in (("wadaptive", "split"), ("wsplit", "split"), ("wadaptive", "wsplit"), ("wadaptive", "adaptive"),
                   ("wadaptive", best_merge), ("wsplit", best_merge), ("wadaptive", "regmean"),
                   ("adaptive", "split"), ("adaptive_fair", "split"), ("kshare", "split"), ("adaptive", "kshare"),
                   ("split", best_merge), ("adaptive", best_merge), ("kshare", best_merge), ("split", "kmerge"),
                   ("wadaptive", "kmerge"), ("adaptive", "kmerge"), (best_merge, "kmerge"), ("linear", "kmerge")):
        if (a_, b_) in [(c_["contrast"].split(" - ")[0], c_["contrast"].split(" - ")[1]) for c_ in cons]:
            continue
        if a_ in mb_ and b_ in mb_ and a_ != b_:
            dm, dw = mb_[a_] - mb_[b_], wb_[a_] - wb_[b_]
            cons.append({"contrast": f"{a_} - {b_}", "d_mean": BOARDS[K].loc[a_, "mean"] - BOARDS[K].loc[b_, "mean"],
                         "mean_95CI": f"[{np.percentile(dm, 2.5):+.3f}, {np.percentile(dm, 97.5):+.3f}]", "p_mean": pval(dm),
                         "d_worst": BOARDS[K].loc[a_, "worst_task"] - BOARDS[K].loc[b_, "worst_task"],
                         "worst_95CI": f"[{np.percentile(dw, 2.5):+.3f}, {np.percentile(dw, 97.5):+.3f}]", "p_worst": pval(dw)})
    CON[K] = pd.DataFrame(cons)
    if len(CON[K]):
        CON[K]["p_mean_holm"] = holm(CON[K].p_mean.values)          # family-wise correction over all listed contrasts
    SIG[K].to_csv(os.path.join(RUN_DIR, f"significance_K{K}.csv"), index=False)
    CON[K].to_csv(os.path.join(RUN_DIR, f"contrasts_K{K}.csv"), index=False)

# ---- capacity stress test (K = 1, slot rank r_s): table + paired bootstrap contrasts for every r_s ----
STRESS_TAB, STRESS_CON = None, {}
try:                                      # the stress test is extra: a problem here must not fail the run
    if os.path.exists(STRESS_CSV):
        ss = pd.read_csv(STRESS_CSV)
        ss = ss[ss.seed.isin(SEEDS) & ss.method.isin(METHOD_LIST)].drop_duplicates(["seed", "r_slot", "method", "task"], keep="last")
        if len(ss):
            _mean = ss.pivot_table(index="method", columns="r_slot", values="nscore", aggfunc="mean")
            _worst = ss.groupby(["method", "r_slot", "seed"]).nscore.min().groupby(["method", "r_slot"]).mean().unstack("r_slot")
            STRESS_TAB = pd.concat({"mean": _mean, "worst_task": _worst}, axis=1)
            STRESS_TAB = STRESS_TAB.sort_values(("mean", min(_mean.columns)), ascending=False)
            _rng = np.random.default_rng(1)
            _idx = {t: _rng.integers(0, len(DATA[t]["test"]), (2000, len(DATA[t]["test"]))) for t in TASKS}
            _solo = {(s, t): stat_score(t, EVAL_CACHE[f"solo|s{s}|{t}"]["stats"], _idx[t]) for s in sorted(set(ss.seed)) for t in TASKS}
            _skey = "+".join(TASKS)
            for rs, g in ss.groupby("r_slot"):
                per = defaultdict(list)
                for r in g.itertuples():
                    m = EVAL_CACHE.get(f"{r.method}|s{r.seed}|r{rs}|{_skey}|{r.task}")
                    if m is None:
                        continue
                    sb, so = stat_score(r.task, m["stats"], _idx[r.task]), _solo[(r.seed, r.task)]
                    per[r.method].append(np.where(so >= 0.05, sb / np.maximum(so, 1e-9), sb))
                mb = {m_: np.mean(v, axis=0) for m_, v in per.items()}
                best_m = max([m_ for m_ in mb if METHODS[m_]["kind"] == "merge"], key=lambda m_: _mean.loc[m_, rs], default=None)
                cons = []
                for a_, b_ in (("wadaptive", "split"), ("adaptive", "split"), ("kshare", "split"), ("wsplit", "split"),
                               ("wadaptive", best_m), ("kshare", best_m), ("split", best_m), ("wadaptive", "kmerge"),
                               ("split", "kmerge")):
                    if a_ in mb and b_ in mb and a_ != b_ and f"{a_} - {b_}" not in [c_["contrast"] for c_ in cons]:
                        dm = mb[a_] - mb[b_]
                        cons.append({"contrast": f"{a_} - {b_}", "d_mean": _mean.loc[a_, rs] - _mean.loc[b_, rs],
                                     "mean_95CI": f"[{np.percentile(dm, 2.5):+.3f}, {np.percentile(dm, 97.5):+.3f}]",
                                     "p_mean": pval(dm)})
                STRESS_CON[rs] = pd.DataFrame(cons)
                if len(STRESS_CON[rs]):
                    STRESS_CON[rs]["p_mean_holm"] = holm(STRESS_CON[rs].p_mean.values)
                    STRESS_CON[rs].to_csv(os.path.join(RUN_DIR, f"stress_contrasts_r{rs}.csv"), index=False)
            STRESS_TAB.to_csv(os.path.join(RUN_DIR, "stress_table.csv"))
except Exception as ex:
    log(f"stress-test analysis skipped: {ex!r}"[:300])
    STRESS_TAB, STRESS_CON = None, {}

# ---------------- figures ----------------
INK, INK2, MUTED, GRID, AXIS, SURF = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
GROUP_COLORS = {"merging (K-Merge)": "#2a78d6", "merging (prior)": "#2a78d6", "sharing (prior idea)": "#1baf7a",
                "sharing (new)": "#eb6834"}
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "axes.edgecolor": AXIS, "axes.labelcolor": INK2,
                     "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
from matplotlib.patches import Patch
Ks = MODE["Ks"]
fig, axs = plt.subplots(1, len(Ks), figsize=(4.6 * len(Ks) + 1.4, 0.34 * len(METHOD_LIST) + 1.6), sharey=False, squeeze=False)
for ax, K in zip(axs[0], Ks):
    b = BOARDS[K]
    yy = np.arange(len(b))[::-1]
    lo_ = max(0.0, float(b["mean"].min()) - 0.08)
    ax.hlines(yy, lo_, b["mean"], color=GRID, lw=1.5, zorder=1)          # dot plot: honest with a zoomed axis
    ax.scatter(b["mean"], yy, s=64, color=[GROUP_COLORS[g] for g in b.group], edgecolor=SURF, linewidth=1.5, zorder=3)
    ax.set_yticks(yy); ax.set_yticklabels(b.index, color=INK)
    for y_, v in zip(yy, b["mean"]):
        ax.text(v + 0.006, y_, f"{v:.3f}", va="center", fontsize=8, color=INK2)
    ax.set_xlim(lo_, max(1.02, float(b["mean"].max()) + 0.06))
    ax.axvline(1.0, color=MUTED, lw=1, ls="--", zorder=0)
    ax.grid(axis="x", color=GRID, lw=0.8, zorder=0)
    ax.set_title(f"K = {K} slots for {len(TASKS)} tasks", loc="left", color=INK, fontsize=10)
    ax.set_xlabel("normalised score (1.0 = own adapter)")
fig.suptitle("Real data: merging vs sharing at the same storage", x=0.01, ha="left", color=INK, fontsize=11)
fig.legend(handles=[Patch(color=GROUP_COLORS[g], label=g) for g in ("merging (prior)", "sharing (prior idea)", "sharing (new)")],
           frameon=False, loc="lower center", ncol=3, fontsize=8)
plt.tight_layout(rect=(0, 0.07, 1, 0.95)); plt.savefig(os.path.join(RUN_DIR, "fig_real_leaderboard.png"), dpi=150); plt.show()

from matplotlib.colors import LinearSegmentedColormap
K0 = Ks[0]
heat = st[st.K == K0].groupby(["method", "task"]).nscore.mean().unstack("task").reindex(index=BOARDS[K0].index, columns=TASKS)
fig, ax = plt.subplots(figsize=(10.5, 0.42 * len(heat) + 1.8))
im = ax.imshow(heat.values, aspect="auto", cmap=LinearSegmentedColormap.from_list(
    "seqblue", ["#0d366b", "#184f95", "#256abf", "#3987e5", "#6da7ec", "#9ec5f4", "#cde2fb"][::-1]))
ax.set_yticks(range(len(heat))); ax.set_yticklabels(heat.index, color=INK)
ax.set_xticks(range(len(TASKS))); ax.set_xticklabels(TASKS, rotation=45, ha="right")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="#ffffff" if v < 0.75 else INK)
ax.set_title(f"Normalised score per task, K = {K0} slots (darker = more of the task's own ability kept)", loc="left",
             color=INK, fontsize=10)
cb = plt.colorbar(im, ax=ax, fraction=0.025); cb.outline.set_visible(False)
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "fig_real_tasks.png"), dpi=150); plt.show()

try:
    if STRESS_TAB is not None:
        rs_list = sorted(_mean.columns)
        STYLE = {"kmerge": ("#2a78d6", "o", "-"), "regmean": ("#2a78d6", "s", "--"), "ties": ("#2a78d6", "^", ":"),
                 "split": ("#1baf7a", "o", "-"), "kshare": ("#eb6834", "s", "--"), "adaptive": ("#eb6834", "^", ":"),
                 "wadaptive": ("#eb6834", "o", "-")}
        fig, ax = plt.subplots(figsize=(7.4, 4.3))
        for m_, (c_, mk_, ls_) in STYLE.items():
            if m_ in _mean.index:
                ax.plot(rs_list, [_mean.loc[m_, r_] for r_ in rs_list], color=c_, marker=mk_, ls=ls_, lw=1.8, ms=6,
                        label=f"{m_} ({METHODS[m_]['group']})")
        ax.set_xscale("log", base=2); ax.set_xticks(rs_list); ax.set_xticklabels([str(r_) for r_ in rs_list])
        ax.set_xlabel(f"directions in the one slot shared by all {len(TASKS)} tasks (r_s)")
        ax.set_ylabel("normalised score (1.0 = own adapter)")
        ax.grid(color=GRID, lw=0.8)
        ax.legend(frameon=False, fontsize=7.5, loc="lower right", handlelength=3.2)
        ax.set_title("Capacity stress test: one slot for every task, fewer directions to the left", loc="left", color=INK,
                     fontsize=10)
        plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "fig_real_stress.png"), dpi=150); plt.show()
except Exception as ex:
    log(f"stress-test figure skipped: {ex!r}"[:300])

# ---------------- summary ----------------
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
q = REF.groupby("task")[["base", "solo"]].mean().loc[TASKS]
lines = ["=" * 100, "SEND THIS BACK", "=" * 100,
         f"REAL BENCHMARK | mode={RUN_MODE} model={MODEL_NAME} seeds={SEEDS} tasks={len(TASKS)} train/test={N_TRAIN}/{N_TEST} "
         f"LoRA r={LORA_R} on {'+'.join(TARGET_MODULES)} K={Ks} orders={MODE['orders']} dtype={str(DTYPE).split('.')[-1]} "
         f"runtime={(time.time() - T0) / 60:.0f} min" + (" | FAKE DATA (pipeline test only)" if FAKE_DATA else "")
         + f" | regmean pinv fallbacks={REGMEAN_FALLBACKS[0]} | whitening ridge retries={WHITEN_FALLBACKS[0]}"
         + (f"\nFAILED (skipped): {FAILED}" if FAILED else ""),
         "", "[1] SINGLE-TASK QUALITY (primary metric: GEC F0.5 / summarization ROUGE-L / QA F1; base = no adapter)",
         q.round(3).to_string()]
for K in Ks:
    lines += ["", f"[2] LEADERBOARD, K = {K} slots (normalised score = task score / own adapter; mean over tasks, seeds, orders)",
              BOARDS[K].round(3).to_string(),
              "", f"[3] SIGNIFICANCE vs K-Merge, K = {K} (paired bootstrap over test examples, 2000 draws, Holm-corrected)",
              SIG[K].round(4).to_string(index=False),
              "", f"[4] KEY CONTRASTS, K = {K} (difference in normalised score, 95% CI, two-sided p)",
              CON[K].round(4).to_string(index=False)]
pp = pd.DataFrame(part_rows)
lines += ["", "[5] WHICH TASKS SHARED A SLOT (K-Merge's cosine rule, same for every method)"]
lines += [f"  seed {s_} K={k_} {o_:9s}: " + "  |  ".join(g_.sort_values("slot").tasks)
          for (s_, k_, o_), g_ in pp.groupby(["seed", "K", "order"])]
try:
    if STRESS_TAB is not None:
        lines += ["", f"[6] CAPACITY STRESS TEST: all {len(TASKS)} tasks in ONE slot that has r_s directions "
                      f"(normalised score; seeds {sorted(set(ss.seed))})", STRESS_TAB.round(3).to_string()]
        for rs_, c_ in STRESS_CON.items():
            lines += ["", f"[6b] STRESS CONTRASTS, r_s = {rs_} (paired bootstrap over test examples; Holm over this list)",
                      c_.round(4).to_string(index=False)]
except Exception as ex:
    lines += ["", f"[6] stress-test summary skipped: {ex!r}"[:300]]
lines += ["=" * 100]
summary = "\n".join(lines)
print(summary)
with open(os.path.join(RUN_DIR, "RESULTS_SUMMARY.txt"), "w", encoding="utf-8") as f:
    f.write(summary)
for K in Ks:
    BOARDS[K].to_csv(os.path.join(RUN_DIR, f"leaderboard_K{K}.csv"))
zpath = os.path.join(OUT_DIR, "results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(RUN_DIR)):
        if fn.endswith((".csv", ".png", ".txt")) and fn != "predictions_test.csv":
            z.write(os.path.join(RUN_DIR, fn), arcname=f"{CFG_TAG}_{RUN_MODE}/{fn}")
log(f"done. Download {zpath} (Output tab) and send it with the summary above.")

## How to read the results

* **[1] Single-task quality:** the normaliser. A task whose adapter barely beats the base model cannot show much merging damage.
* **[2] Leaderboard:** 1.0 means every task kept all of its own adapter's ability. `K = 2` puts about 6 tasks in each slot
  (hard), `K = 4` about 3. `worst_task` is the task that lost the most, averaged over seeds and orders.
* **[3] Significance:** a difference is real only if the Holm-corrected p is below 0.05.
* **[4] Key contrasts:** `wadaptive - split`, `wsplit - split`, `adaptive - split` and `kshare - split` decide whether a NEW
  sharing method beats the existing idea; `wadaptive - <best merge>` and `split - <best merge>` decide whether sharing beats
  the strongest merging method on real data (`wsplit`/`wadaptive` use the same 32 unlabeled inputs per task as RegMean).
  `p_mean_holm` is corrected over all listed contrasts.
* **[5] Slots:** which tasks K-Merge's rule put together (by language, by problem type, or mixed).
* **[6] Capacity stress test:** every task in ONE slot with only r_s directions (r_s = 4 means a third of a direction per
  task). This is where a smarter way to share the slot can matter; `[6b]` tests it per r_s.